# Lesson 02 — A Mini Masked Autoencoder (and why pixel targets disappoint)

**Read first:** `docs/03_from_mae_to_ijepa.md`, sections 3.1–3.2

You will build MAE (He et al., 2021) on static MNIST: mask most of the image, encode only visible patches, reconstruct the missing **pixels** with a small decoder. Then you will measure its representation with a probe and *see with your own eyes* the two problems that motivated JEPA:

1. reconstructions go blurry where the answer is ambiguous (L2 → conditional mean),
2. lower pixel loss stops implying better features.

The architecture skeleton (encoder on visible tokens + light decoder with mask tokens) is **identical** to V-JEPA's — only the target space will change in lesson 03. Time: ~15 min of training on a T4.

In [ ]:
#@title Setup — run this first { display-mode: "form" }
# Works both on Colab (clones the repo) and locally (repo already present).
import os, sys

if os.path.exists("../src/vjepa_mini"):          # running locally from notebooks/
    sys.path.insert(0, os.path.abspath("../src"))
elif os.path.exists("world-model-jepa/src"):      # already cloned on Colab
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))
else:                                             # fresh Colab runtime
    REPO_URL = "https://github.com/josephlionel95-moon/world-model-jepa.git"
    os.system(f"git clone {REPO_URL}")
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("NOTE: Runtime > Change runtime type > T4 GPU for the training notebooks.")


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from vjepa_mini.models.vit import Block          # reuse OUR transformer block
from vjepa_mini.models.patch_embed import sincos_1d
from vjepa_mini.utils import set_seed

set_seed(0)
# MNIST 28x28, patch 4 -> 7x7 = 49 tokens of 16 pixels each
PATCH, GRID, N_TOK, DIM = 4, 7, 49, 128

tfm = transforms.ToTensor()
train_ds = datasets.MNIST("./mnist", train=True, download=True, transform=tfm)
test_ds  = datasets.MNIST("./mnist", train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, drop_last=True)
print(len(train_ds), "train images")

## The model

MAE's asymmetry is the point: a real encoder that sees only visible patches, and a *throwaway* decoder that gets mask tokens and paints pixels. Keep the decoder small — its job is to be barely good enough that the encoder must do the understanding.

In [ ]:
def sincos_2d(dim, grid):
    """2D position codes: concat h-code and w-code (image version of the 3D one)."""
    half = dim // 2
    h = sincos_1d(half, torch.arange(grid))
    w = sincos_1d(dim - half, torch.arange(grid))
    pe = torch.cat([
        h[:, None, :].expand(grid, grid, half),
        w[None, :, :].expand(grid, grid, dim - half),
    ], dim=-1)
    return pe.reshape(grid * grid, dim)

class MiniMAE(nn.Module):
    def __init__(self, dim=DIM, depth=4, dec_dim=64, dec_depth=2, heads=4):
        super().__init__()
        self.patchify = nn.Linear(PATCH * PATCH, dim)          # 16 pixels -> dim
        self.register_buffer("pe", sincos_2d(dim, GRID).unsqueeze(0))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        # ---- decoder (discarded after pretraining) ----
        self.dec_embed = nn.Linear(dim, dec_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dec_dim))
        self.register_buffer("dec_pe", sincos_2d(dec_dim, GRID).unsqueeze(0))
        self.dec_blocks = nn.ModuleList([Block(dec_dim, 4) for _ in range(dec_depth)])
        self.dec_out = nn.Linear(dec_dim, PATCH * PATCH)       # back to raw pixels

    def to_patches(self, img):                                  # [B,1,28,28] -> [B,49,16]
        B = img.shape[0]
        p = img.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH)
        return p.permute(0, 2, 3, 1, 4, 5).reshape(B, N_TOK, -1)

    def encode(self, img, keep_idx=None):
        x = self.patchify(self.to_patches(img)) + self.pe       # position FIRST
        if keep_idx is not None:
            x = x[:, keep_idx]                                  # THEN drop
        for b in self.blocks: x = b(x)
        return self.norm(x)

    def forward(self, img, keep_idx, mask_idx):
        z = self.encode(img, keep_idx)
        d = self.dec_embed(z) + self.dec_pe[:, keep_idx]
        m = self.mask_token.expand(img.shape[0], len(mask_idx), -1) + self.dec_pe[:, mask_idx]
        x = torch.cat([d, m], dim=1)
        for b in self.dec_blocks: x = b(x)
        pred = self.dec_out(x[:, len(keep_idx):])               # only mask positions
        target = self.to_patches(img)[:, mask_idx]
        return F.mse_loss(pred, target), pred

mae = MiniMAE().to(device)
print(f"{sum(p.numel() for p in mae.parameters())/1e6:.2f}M params")

**Before training, predict:** we will mask 75% of patches uniformly at random. MNIST digits occupy ~20% of the canvas. Will the model learn to reconstruct digits at all? What will reconstructions look like when the visible 25% is compatible with several digits?

In [ ]:
MASK_RATIO = 0.75
opt = torch.optim.AdamW(mae.parameters(), lr=1.5e-3, weight_decay=0.05)
EPOCHS = 3   # ~4 min on T4

for epoch in range(EPOCHS):
    for i, (img, _) in enumerate(train_dl):
        img = img.to(device)
        perm = torch.randperm(N_TOK)
        n_keep = int(N_TOK * (1 - MASK_RATIO))
        keep_idx, mask_idx = perm[:n_keep], perm[n_keep:]
        loss, _ = mae(img, keep_idx, mask_idx)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if i % 100 == 0:
            print(f"epoch {epoch} step {i:>3} | pixel L2 {loss.item():.4f}")

## Look at the reconstructions

The interesting failure is not "it can't reconstruct" — it's *how* it fails when the context is ambiguous.

In [ ]:
@torch.no_grad()
def show_reconstructions(n=8, ratio=MASK_RATIO):
    img = torch.stack([test_ds[i][0] for i in range(n)]).to(device)
    perm = torch.randperm(N_TOK)
    n_keep = int(N_TOK * (1 - ratio))
    keep_idx, mask_idx = perm[:n_keep], perm[n_keep:]
    _, pred = mae(img, keep_idx, mask_idx)
    # stitch: visible patches from input, masked patches from prediction
    patches = mae.to_patches(img).clone()
    patches[:, mask_idx] = pred
    recon = patches.reshape(n, GRID, GRID, PATCH, PATCH).permute(0,1,3,2,4).reshape(n,28,28)
    visible = mae.to_patches(img).clone(); visible[:, mask_idx] = 0.5
    vis = visible.reshape(n, GRID, GRID, PATCH, PATCH).permute(0,1,3,2,4).reshape(n,28,28)
    fig, axes = plt.subplots(3, n, figsize=(1.3*n, 4))
    for j in range(n):
        axes[0,j].imshow(img[j,0].cpu(), cmap="gray"); axes[0,j].set_axis_off()
        axes[1,j].imshow(vis[j].cpu(), cmap="gray");    axes[1,j].set_axis_off()
        axes[2,j].imshow(recon[j].cpu().clamp(0,1), cmap="gray"); axes[2,j].set_axis_off()
    axes[0,0].set_title("input", loc="left"); axes[1,0].set_title("visible (75% masked)", loc="left")
    axes[2,0].set_title("reconstruction", loc="left")
    plt.tight_layout(); plt.show()

show_reconstructions()
show_reconstructions(ratio=0.9)   # crank ambiguity up
# Note the soft, averaged strokes where several completions were possible:
# L2 loss -> the model outputs the conditional MEAN of all compatible digits.

## Probe the representation

Now the question that actually matters: what did the **encoder** learn? Freeze it, average-pool its tokens, train logistic regression on digit classes. (A linear probe is enough for MNIST-scale; we graduate to attentive probes in lesson 05.)

In [ ]:
@torch.no_grad()
def features(ds, n=10000):
    xs, ys = [], []
    dl = DataLoader(ds, batch_size=512)
    seen = 0
    for img, y in dl:
        z = mae.encode(img.to(device)).mean(dim=1)   # [B, DIM] mean-pooled
        xs.append(z.cpu()); ys.append(y); seen += len(y)
        if seen >= n: break
    return torch.cat(xs)[:n], torch.cat(ys)[:n]

def linear_probe(xtr, ytr, xte, yte, epochs=30):
    clf = nn.Linear(xtr.shape[1], 10).to(device)
    o = torch.optim.AdamW(clf.parameters(), lr=1e-2)
    xtr, ytr = xtr.to(device), ytr.to(device)
    for _ in range(epochs):
        o.zero_grad(); F.cross_entropy(clf(xtr), ytr).backward(); o.step()
    with torch.no_grad():
        acc = (clf(xte.to(device)).argmax(1).cpu() == yte).float().mean().item()
    return acc

xtr, ytr = features(train_ds); xte, yte = features(test_ds, 5000)
acc = linear_probe(xtr, ytr, xte, yte)
print(f"MAE frozen linear-probe accuracy: {acc:.3f}   (chance = 0.10)")
# Save for comparison in lesson 03:
torch.save({"acc": acc}, "mae_probe_result.pt")

## The experiment that motivates everything: masking ratio sweep

The paper-scale fact (MAE paper, Fig. 5): representation quality peaks at surprisingly *high* masking. Reproduce it in miniature. This takes ~15 min — start it and read `docs/03` section 3.3 while it runs.

In [ ]:
def train_mae(ratio, epochs=2):
    set_seed(0)
    m = MiniMAE().to(device)
    o = torch.optim.AdamW(m.parameters(), lr=1.5e-3, weight_decay=0.05)
    for _ in range(epochs):
        for img, _ in train_dl:
            img = img.to(device)
            perm = torch.randperm(N_TOK)
            k = max(1, int(N_TOK * (1 - ratio)))
            loss, _ = m(img, perm[:k], perm[k:])
            o.zero_grad(set_to_none=True); loss.backward(); o.step()
    global mae; mae = m
    xtr, ytr = features(train_ds, 6000); xte, yte = features(test_ds, 3000)
    return linear_probe(xtr, ytr, xte, yte)

ratios = [0.25, 0.5, 0.75, 0.9]
accs = [train_mae(r) for r in ratios]
plt.plot(ratios, accs, "o-"); plt.xlabel("masking ratio"); plt.ylabel("probe accuracy")
plt.title("Harder pretext task -> better features (up to a point)"); plt.grid(alpha=0.3); plt.show()
for r, a in zip(ratios, accs): print(f"ratio {r:.2f}: acc {a:.3f}")

## What to take away

- **Blur = conditional mean.** Where the future/hidden content is ambiguous, an L2 pixel model *must* hedge. Capacity is spent on hedging, not on understanding. (Write down: what would the model output at a masked location if it were trained with L1 pixels instead? Median — try it, exercise 2.)
- **The pixel loss is not the metric you care about.** Watch: pixel loss falls monotonically while probe accuracy saturates. In lesson 03, the target space becomes learned — and then the loss becomes *actively misleading*, which is why we probe.
- **The skeleton survives.** Encoder-on-visible + mask-tokens-in-a-light-head is exactly V-JEPA's forward pass. In lesson 03 we delete the pixel decoder and point the predictor at features.

## Exercises

1. Masking *structure*: replace uniform-random with block masking (mask one contiguous 5x5 block of patches). Same ratio ~50%. Compare probe accuracy — you're reproducing I-JEPA's key masking insight one lesson early.
2. Swap the decoder loss to L1. Look at reconstructions: sharper or just different? Why? (Median vs mean of the conditional pixel distribution.)
3. Give the decoder 4x more depth. Does pixel loss improve? Does probe accuracy? What does the divergence tell you about where understanding lives?
4. Challenge: probe accuracy vs *encoder layer* — which layer's mean-pooled features probe best, and why might the last layer NOT be optimal for a reconstruction-trained model?

Next: `03_ijepa_mini_images.ipynb` — same skeleton, learned targets, and your first encounter with collapse.